# -- Notebook này là Backend phục vụ cho việc sinh hình ảnh cho AI Adventure --

In [1]:
!pip install ngrok pyngrok

In [2]:
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import asyncio
import torch
import base64
import os
import re
from io import BytesIO
from typing import Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, MarianMTModel, MarianTokenizer
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerAncestralDiscreteScheduler
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import google.generativeai as genai
from pyngrok import ngrok

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


In [3]:
# 1. Cấu hình môi trường
nest_asyncio.apply()
app = FastAPI(title="AI Adventure Unified API")

# --- CẤU HÌNH API KEYS ---

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GENAI_API_KEY = user_secrets.get_secret("YOUR_GEMINI_API_KEY")
NGROK_TOKEN = user_secrets.get_secret("YOUR_NGROK_TOKEN")
genai.configure(api_key=GENAI_API_KEY)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- KHỞI TẠO LOCAL MODELS (Lazy Loading để tiết kiệm VRAM ban đầu) ---

In [7]:
print("Đang chuẩn bị các mô hình local...")

Đang chuẩn bị các mô hình local...


In [8]:
# 1. Local Summarizer: Qwen 2.5 1.5B
qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"
q_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
q_model = AutoModelForCausalLM.from_pretrained(qwen_name, torch_dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
# 2. Local Translator: MarianMT
trans_name = "Helsinki-NLP/opus-mt-vi-en"
t_tokenizer = MarianTokenizer.from_pretrained(trans_name)
t_model = MarianMTModel.from_pretrained(trans_name).to(device)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [10]:
# 3. SDXL Lightning
base_model = "stabilityai/stable-diffusion-xl-base-1.0"
repo = "ByteDance/SDXL-Lightning"
unet = UNet2DConditionModel.from_config(base_model, subfolder="unet").to(device, torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo, "sdxl_lightning_4step_unet.safetensors"), device=device))
pipe = StableDiffusionXLPipeline.from_pretrained(base_model, unet=unet, torch_dtype=torch.float16, variant="fp16").to(device)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:250: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.load_config(...) followed by <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.from_config(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

sdxl_lightning_4step_unet.safetensors:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

# --- CÁC HÀM TRỢ GIÚP ---

In [11]:
def sanitize_filename(name: str) -> str:
    # Chuyển thành chữ thường, thay khoảng trắng bằng gạch dưới, xóa ký tự đặc biệt
    name = name.lower().strip()
    name = re.sub(r'[^\w\s-]', '', name)
    return re.sub(r'[-\s]+', '_', name)

async def summarize_with_gemini(text: str) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')
    
    # Prompt được thiết kế chuyên biệt cho việc trích xuất từ khóa thị giác
    prompt = f"""
    You are an expert Prompt Engineer for Stable Diffusion.
    Your task is to analyze the following fantasy world lore (written in Vietnamese) and extract ONLY the visual elements.
    
    Rules:
    1. DO NOT write a story or full sentences.
    2. Focus ONLY on: landscape, architecture, weather, lighting, atmosphere, and key physical objects.
    3. Output the result as a comma-separated list of keywords in ENGLISH.
    
    Example Output Format:
    ruined gothic city, heavy ash falling, dark cloudy sky, faint glowing magic traps, gloomy atmosphere, highly detailed
    
    World Lore to analyze:
    {text}
    """
    response = await model.generate_content_async(prompt)
    return response.text.strip()

def summarize_with_qwen(text: str) -> str:
    # Prompt tiếng Việt trực diện, dễ hiểu cho model 1.5B
    system_instruction = "Bạn là trợ lý phân tích hình ảnh. Hãy trích xuất các từ khóa về phong cảnh, thời tiết, và kiến trúc từ đoạn văn sau. Không kể chuyện, chỉ liệt kê các cụm từ ngắn gọn."
    user_input = f"Đoạn văn: {text}\n\nTừ khóa thị giác:"
    
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_input}
    ]
    
    # 1. Bật return_dict=True để lấy cấu trúc chuẩn (gồm input_ids và attention_mask)
    inputs = q_tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True  # <--- Quan trọng để tránh lỗi định dạng Tensor
    ).to(device)
    
    # 2. Sửa lỗi trong hàm generate
    outputs = q_model.generate(
        **inputs,  # Giải nén cả input_ids và attention_mask vào model
        max_new_tokens=60, 
        temperature=0.3, 
        repetition_penalty=1.2,
        do_sample=True,  # <--- Bắt buộc phải có khi dùng temperature
        pad_token_id=q_tokenizer.eos_token_id  # Tránh warning thiếu pad_token
    )
    
    # 3. Tính toán độ dài chuẩn để cắt bỏ phần prompt ban đầu
    input_length = inputs["input_ids"].shape[1]
    summary_vi = q_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    
    # Đưa chuỗi từ khóa tiếng Việt qua MarianMT để dịch sang tiếng Anh
    t_inputs = t_tokenizer(summary_vi, return_tensors="pt", padding=True).to(device)
    t_tokens = t_model.generate(**t_inputs)
    en_tags = t_tokenizer.decode(t_tokens[0], skip_special_tokens=True)
    
    return en_tags.strip()

# --- ENDPOINT CHÍNH ---

In [12]:
class UnifiedRequest(BaseModel):
    prompt: str
    world_name: str = "default_world"
    summarize_model: str # "gemini-2.5-flash" hoặc "Qwen/Qwen2.5-1.5B-Instruct"
    txt2img_model: str # "SDXL lightning" hoặc "Gemini API banana pro"
    step: int = 4

@app.post("/generate-world-theme")
async def generate_world_theme(request: UnifiedRequest):
    try:
        # Tự động phát hiện nếu prompt đã là tiếng Anh (không chứa các dấu thanh tiếng Việt)
        # để bỏ qua bước tóm tắt & dịch thuật nhằm giữ nguyên mô tả nhân vật tiếng Anh chi tiết.
        has_vietnamese = bool(re.search(
            r'[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệđìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ]', 
            request.prompt, 
            re.IGNORECASE
        ))
        
        if not has_vietnamese:
            # Prompt đã bằng tiếng Anh, giữ nguyên
            en_prompt = request.prompt.strip()
        else:
            # Bước 1: Tóm tắt & Dịch thuật (cho các prompt tiếng Việt)
            if "gemini" in request.summarize_model.lower():
                en_prompt = await summarize_with_gemini(request.prompt)
            else:
                en_prompt = summarize_with_qwen(request.prompt)
        
        # Tự động nhận diện xem đây là ảnh chân dung (Avatar) hay cảnh thế giới (See the World)
        # dựa vào từ khóa trong prompt hoặc tên file đầu ra.
        is_portrait = any(
            k in en_prompt.lower() or k in request.world_name.lower() 
            for k in ["avatar", "portrait", "character", "monk", "merchant", "lemuen", "hoshiguma", "wang", "vinavictoria"]
        )
        
        if is_portrait:
            # Prompt phụ trợ chuyên cho vẽ chân dung nhân vật RPG
            final_prompt = f"{en_prompt}, close up portrait, headshot, character design, fantasy, masterpiece, highly detailed, cinematic lighting"
        else:
            # Prompt phụ trợ chuyên cho vẽ phong cảnh thế giới
            final_prompt = f"{en_prompt}, masterpiece, highly detailed, fantasy concept art, cinematic lighting"
            
        # Bước 2: Sinh hình ảnh
        if "sdxl" in request.txt2img_model.lower():
            image = pipe(final_prompt, num_inference_steps=request.step, guidance_scale=0).images[0]
        else:
            raise HTTPException(status_code=501, detail="Gemini Image API chưa được cấu hình cụ thể")

        # Bước 3: Lưu hình ảnh xuống Local (Kaggle Storage)
        save_dir = "generated_imgs/theme_stories"
        os.makedirs(save_dir, exist_ok=True)
        
        safe_name = sanitize_filename(request.world_name)
        file_path = os.path.join(save_dir, f"{safe_name}_theme.png")
        
        image.save(file_path) # Tự động ghi đè nếu trùng tên

        # Bước 4: Chuyển sang Base64 để trả về API
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()

        return {
            "status": "success",
            "en_prompt": final_prompt,
            "saved_at": file_path,
            "image_base64": img_str
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# --- KHỞI CHẠY NGROK & SERVER ---

In [13]:
async def start_server():
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"\n🌍 API ĐÃ PUBLIC: {public_url}/docs")
    
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

asyncio.create_task(start_server())

<Task pending name='Task-1' coro=<start_server() running at /tmp/ipykernel_186/3341530261.py:1>>

                                                                                                    
🌍 API ĐÃ PUBLIC: NgrokTunnel: "https://onion-vertical-squash.ngrok-free.dev" -> "http://localhost:8000"/docs


INFO:     Started server process [186]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     14.169.12.124:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.169.12.124:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     14.169.12.124:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.169.12.124:0 - "GET /openapi.json HTTP/1.1" 200 OK


  0%|          | 0/8 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

INFO:     14.169.12.124:0 - "POST /generate-world-theme HTTP/1.1" 200 OK


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [', mysterious atmosphere, tense atmosphere, oppressive silence, expectant silence, lingering calamity, leaden air, charcoal dust in air, ruined statues, featureless faces, empty eye sockets, unnatural ash spiral patterns, distant horizon, highly detailed, masterpiece, highly detailed, fantasy concept art, cinematic lighting']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [', mysterious atmosphere, tense atmosphere, oppressive silence, expectant silence, lingering calamity, leaden air, charcoal dust in air, ruined statues, featureless faces, empty eye sockets, unnatural ash spiral patterns, distant horizon, highly detailed, masterpiece, highly detailed, fantasy concept art, cinematic lighting']


  0%|          | 0/4 [00:00<?, ?it/s]

INFO:     117.5.142.209:0 - "POST /generate-world-theme HTTP/1.1" 200 OK


# Testing